# Import libraries

In [1]:
from langgraph.graph import StateGraph, START, END
import logging
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage, BaseMessage
from typing import TypedDict, List, Optional
from pydantic import BaseModel, Field
from typing import Dict, Optional, Literal
from dotenv import load_dotenv

c:\Users\amit7\OneDrive\Desktop\AudioBot\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
load_dotenv()

False

In [3]:
logger = logging.getLogger(__name__)

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.8,
    max_tokens=None,
    timeout=None,
    max_retries=2,
)

GroqError: The api_key client option must be set either by passing api_key to the client or by setting the GROQ_API_KEY environment variable

In [ ]:
# Create AgentState:
class AgentState(TypedDict):
    # user input for current turn
    user_input: str

    # full conversation history
    conversation: List[BaseMessage]

    # system message instruction
    system_message: Optional[str]

    # conversation/session id
    conversation_id: Optional[str]

    # classified intent (e.g., chat, tool, clarify)
    intent: Optional[str]

    # structured response parts
    acknowledgement: Optional[str]
    analysis: Optional[dict]

    # final response returned to client
    output: str

In [ ]:
def chat_node(state: AgentState) -> AgentState:
    messages = []

    # Only add system message on first turn
    if not state.get("conversation"):
        system = state.get("system_message") or "You are a helpful assistant."
        messages.append(SystemMessage(content=system))

    # Conversation history (already includes SystemMessage after turn 1)
    messages.extend(state.get("conversation", []))

    # Current user input
    messages.append(HumanMessage(content=state["user_input"]))

    response = llm.invoke(messages)

    return {
        "output": response.content,
        "conversation": messages + [AIMessage(content=response.content)]
    }

In [ ]:
from redis import Redis
from langgraph.checkpoint.redis import RedisSaver

REDIS_URL = "redis://localhost:6379"

# Option 1 — pass URL string directly ✅
redis_saver = RedisSaver(redis_url=REDIS_URL)
redis_saver.setup()

In [ ]:
def build_agent():
    graph = StateGraph(AgentState)
    graph.add_node("chat", chat_node)
    graph.add_edge(START, "chat")
    graph.add_edge("chat", END)
    return graph.compile(checkpointer=redis_saver)
g = build_agent()

In [ ]:
SYSTEM_MESSAGE="""You are an HR interviewer assessing cultural fit. 
Begin by understanding the candidate’s background: personal overview, family background, and educational journey.
Ask one question at a time. 
After gathering this context, ask relevant follow-up questions based on their responses to assess personality, values, teamwork, communication style, adaptability, accountability, and conflict handling. 
Use behavioral and situational questions when appropriate. If answers are vague, ask clarifying follow-ups. Maintain a professional and neutral tone. 
Do not provide feedback or evaluations during the interview."""

In [ ]:
def chat(user_id: str = "user_1"):
    config = {"configurable": {"thread_id": user_id}}
    print("HR Interview started! Type 'quit' to exit.\n")
    
    # First turn — trigger the interviewer to open the conversation
    result = g.invoke({
        "user_input": "Hello, I am ready for the interview.",
        "system_message": SYSTEM_MESSAGE
    }, config=config)
    print(f"Interviewer: {result['output']}\n")

    while True:
        user_input = input("You: ").strip()

        if user_input.lower() in ["quit", "exit", "bye"]:
            print("Interviewer: Thank you for your time. We will be in touch!")
            break

        if not user_input:
            continue

        result = g.invoke({
            "user_input": user_input,
            "system_message": SYSTEM_MESSAGE  # pass every turn
        }, config=config)

        print(f"Interviewer: {result['output']}\n")

In [ ]:
chat('djdj')

HR Interview started! Type 'quit' to exit.

Interviewer: Welcome to the interview, I'm looking forward to speaking with you. To start, could you tell me a little bit about your personal background, including where you're from, your family, and any interesting experiences that have shaped who you are today?

Interviewer: That's a nice observation about the weather in Philadelphia. However, I'd like to learn more about you. Let's get back to the interview. Can you tell me a bit about your educational journey, such as where you went to school, what you studied, and any notable achievements or experiences you had during that time?

Interviewer: It seems like there might have been a misunderstanding. Could you please provide a more detailed and coherent response? To clarify, I'm asking about your educational background, such as the schools you attended, the subjects you studied, and any notable experiences or achievements you had during that time. Please feel free to share as much or as lit